# modular-diffusion: the three-band assembly, part 2: random starts in the bands that read the prompt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.11.0/colab/three_band_part2.ipynb)

Part 1 (`colab/three_band.ipynb`) made the MIDDLE and NOISY band engines of Stable Diffusion 1.5 with the model's own weights inherited. This notebook asks whether inheritance is the lever in the bands that read the prompt: the shape `tiny` (one layer a stage, no middle block, three stages) from RANDOM weights, taught by the model's answers and reading the prompt, on the middle band (steps 291 to 809) and on the noisy band (steps 691 to 999), seeds 0 and 1, by part 1's recipe and data: 3,000 steps at batch 16, pure Adam, an averaged copy used for every read, read every 250 steps on 64 held-out rows.

Each engine is read in the sampler beside the model (the change of the final latent against the model's own sample); each band's seed-0 engine is judged by its PICTURES on 500 of part 1's held-out prompts, against the model's own pictures and the real pictures (KID, Frechet, the CLIP text-to-picture score), with part 1's floor. The model's own 1,000 pictures and the real pictures are brought back from part 1's results on the Hub (they run here only when they are not there).

1. install at a release tag and print the environment;
2. the model and both COCO val2017 parquet files at pinned revisions; the probe, judge and training latents (as part 1); the CLIP judge; a smoke run of every job kind of this list on a tiny random model, with its checks; the real judges on 16 pictures of the real model, with the judge's own control; the smoke shipped; the license copy and the notice shipped only when they are not already in the results folder; a timing of `tiny`; the job list with its projected minutes;
3. the job list, one job at a time; each finished job folder is shipped at once; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

WEIGHTS: the trained engines are derivatives of Stable Diffusion 1.5; they are stored beside the tables in the results repository with a copy of that model's license and a notice at the root of the results folder (shipped by part 1; this notebook ships them only if they are missing). The checkpoint used for resuming is not shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`. On a new machine, finished jobs are fetched back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard,judge] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.11.0"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import numpy as np
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 11, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/three_band_part2"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

As part 1: the model at a pinned revision, both validation parquet files of COCO 2017 at the same revisions, the PROBE (the first 256 pictures of the first file, its content hash checked; the reads use its first 64 rows), the JUDGE latents (the first 1,000 pictures) and the TRAINING latents (the other 4,000), each written once and skipped when present, and the CLIP model at a pinned revision. No engine of an earlier stage is needed: the clean band is served by the model itself. Then every job kind of this list runs for a few steps on a tiny random model (an engine exchanged for itself must change nothing, and every training writes its read curve); the real judges run on 16 pictures of the real model with the judge's own control; the smoke folder is shipped; the license copy and the notice are shipped if the results folder does not hold them yet; `tiny` is timed; the job list is sized from that time.

In [ ]:
# C2 - the model, the data, the inputs, the checks, the size of the job list
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture import BANDS, pictures
from geolip.modular_diffusion.capture.assemble import Assembly
from geolip.modular_diffusion.capture.data import (LatentBatches, center_crop, encode_shards, parquet_pictures, read_index,
                                                   rows_in_order, synthetic_shards)
from geolip.modular_diffusion.capture.engine import load_engine
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.capture.train import train_band_engine
from geolip.modular_diffusion.experiments import capture
TEACHER, PROBE, JUDGE, TRAIN, RUNS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/judge", f"{OUT}/train", f"{OUT}/runs"
CLIP = f"{OUT}/clip"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/capture/three_band"
NOT_SHIPPED = ("checkpoint.pt", "SHIPPED", "FROM_HUB")           # the resume checkpoint and two markers; the rest ships
os.makedirs(RUNS, exist_ok=True)

# the data
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUETS = [hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                            filename=f"data/val-0000{i}-of-00002.parquet", local_dir=f"{OUT}/coco") for i in (0, 1)]
N_PROBE, N_JUDGE = 256, 1000

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

def first_captions(path):
    out = []        # the first caption of every row, read without the pictures (the rule of parquet_pictures)
    for b in pq.ParquetFile(path).iter_batches(batch_size=512, columns=["answer"]):
        for a in b.column(0).to_pylist():
            out.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    return out

EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"           # the 256 probe pictures of the band-needs run and the first capture

def slices(sources):
    # 512 pictures at a time (decoded, handed to the encoder one by one, then dropped), so memory stays flat
    for path, first_row, end in sources:
        total = pq.ParquetFile(path).metadata.num_rows if end is None else end
        for first in range(first_row, total, 512):
            pics, _, _ = parquet_pictures(path, min(512, total - first), skip=first)
            yield from pics
            del pics

JUDGE_SOURCES = [(PARQUETS[0], 0, N_JUDGE)]
TRAIN_SOURCES = [(PARQUETS[0], N_JUDGE, None), (PARQUETS[1], 0, None)]
captions_of = lambda sources: [c for path, first, end in sources for c in first_captions(path)[first:end]]
judge_captions, train_captions = captions_of(JUDGE_SOURCES), captions_of(TRAIN_SOURCES)
JUDGE_ROWS, TRAIN_ROWS = len(judge_captions), len(train_captions)
print("judge pictures:", JUDGE_ROWS, "| expected 1,000; training pictures:", TRAIN_ROWS, "| expected 4,000")
assert JUDGE_ROWS == N_JUDGE and TRAIN_ROWS == 4000, "the judge or training rows are not the expected ones"
ready = lambda folder, n: os.path.exists(f"{folder}/index.json") and sum(read_index(folder)["counts"]) == n
probe_ready, judge_ready, train_ready = ready(PROBE, N_PROBE), ready(JUDGE, JUDGE_ROWS), ready(TRAIN, TRAIN_ROWS)
if not (probe_ready and judge_ready and train_ready):
    with_vae = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    if not probe_ready:
        pictures_, captions, meta = parquet_pictures(PARQUETS[0], N_PROBE)
        HASH = content_hash(pictures_, captions)
        print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
        assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
        encode_shards(pictures_, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device="cuda")
        json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
                  open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
        del pictures_
    if not judge_ready:
        # encode_shards is given a generator of pictures; a stopped encoding continues at its first unfinished shard
        encode_shards(slices(JUDGE_SOURCES), judge_captions, JUDGE, with_vae.vae, with_vae.tokenize, shard_size=500, batch=16,
                      size=512, device="cuda")
    if not train_ready:
        encode_shards(slices(TRAIN_SOURCES), train_captions, TRAIN, with_vae.vae, with_vae.tokenize, shard_size=512, batch=16,
                      size=512, device="cuda")
    del with_vae
    gc.collect()
    torch.cuda.empty_cache()
assert sum(read_index(JUDGE)["counts"]) == N_JUDGE and sum(read_index(TRAIN)["counts"]) == TRAIN_ROWS
print("probe shards:", read_index(PROBE)["counts"], "| judge shards:", read_index(JUDGE)["counts"], "| training shards:",
      read_index(TRAIN)["counts"])

# the job list: (tag, arguments, the jobs it waits for, kind); the two reference picture jobs of part 1 stay in the list
# so that the third cell's fetch brings them back from the Hub (a missing one runs here)
COMMON = (f"teacher={TEACHER} device=cuda threads=4 probe_data={PROBE} probe_first=0 probe_rows=64 probe_seed=1000 "
          "probe_per_band=4 chunk=16")
TRAINING = f"variant=band_engine init=random text=1 batch=16 lr=5e-5 autocast=bf16 read_every=250 keep_checkpoint=0 data={TRAIN} dtype=fp32"
DISTANCE = "dtype=fp16 sample_steps=50 guidance=7.5 group=16"     # the teacher in half precision; the engines stay float32
PICTURES = (f"variant=pictures dtype=fp16 part_dtype=fp16 pic_data={JUDGE} pic_first=0 sample_steps=50 guidance=7.5 "
            f"judge=inception clip={CLIP} sheet_n=16")
TEACHER_PARTS = "clean=teacher,middle=teacher,noisy=teacher"
BANDS2 = ("middle", "noisy")

def job_list(runs, rows=(1000, 500, 100), blocks=10, group=50):
    at = lambda t: f"{runs}/{t}"
    big, half, small = rows
    pic = f"{PICTURES} group={group}"
    cmp = f"variant=picture_compare blocks={blocks}"
    jobs = [("pic_rows_1000", f"{pic} source=rows pic_n={big}", [], "pictures"),
            ("pic_teacher_1000", f"{pic} parts={TEACHER_PARTS} pic_n={big} noise_seed=2000", [], "pictures")]
    for band in BANDS2:
        for k in (0, 1):
            jobs.append((f"{band}_tiny_random_s{k}", f"{TRAINING} shape=tiny band={band} steps=3000 seed={k}", [], "train"))
    for band in BANDS2:
        for k in (0, 1):
            eng = f"{band}_tiny_random_s{k}"
            parts = f"clean=teacher,middle={at(eng)},noisy=teacher" if band == "middle" else f"clean=teacher,middle=teacher,noisy={at(eng)}"
            jobs.append((f"asm_{band}_tiny_random_s{k}", f"variant=assembly parts={parts} {DISTANCE}", [eng], "sample"))
    for band in BANDS2:
        eng = f"{band}_tiny_random_s0"
        parts = f"clean=teacher,middle={at(eng)},noisy=teacher" if band == "middle" else f"clean=teacher,middle=teacher,noisy={at(eng)}"
        jobs.append((f"pic_{band}_tiny_random_s0_500", f"{pic} parts={parts} pic_n={half} noise_seed=2000", [eng], "pictures"))
    for band in BANDS2:
        x = f"{band}_tiny_random_s0_500"
        jobs.append((f"cmp_{x}_vs_teacher", f"{cmp} a={at('pic_' + x)} b={at('pic_teacher_1000')}", ["pic_" + x, "pic_teacher_1000"], "compare"))
        jobs.append((f"cmp_{x}_vs_rows", f"{cmp} a={at('pic_' + x)} b={at('pic_rows_1000')}", ["pic_" + x, "pic_rows_1000"], "compare"))
    for t, args, needs, kind in jobs:    # every folder a job reads is a job before it
        assert all(n in [j[0] for j in jobs[:jobs.index((t, args, needs, kind))]] for n in needs), t
    return jobs

# the text-to-picture judge at a pinned revision: its settings and weights only
snapshot_download(repo_id="openai/clip-vit-large-patch14", revision="32bd64288804d66eefd0ccbe215aa642df71cc41",
                  allow_patterns=["config.json", "model.safetensors", "preprocessor_config.json"], local_dir=CLIP)

# the smoke: every job kind of the list on the tiny random model, a few steps each, with the checks that apply; the
# list needs no engine of an earlier stage
DEVICE = "cuda"
SMOKE = f"{OUT}/smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/rows", rows=8, shard_size=3, size=16, seed=4)       # the tiny model's 16 x 16 latents
synthetic_shards(f"{SMOKE}/judge", rows=20, shard_size=7, size=16, seed=5)
SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP = (20, 10, 4), 2, 2      # at least 2 rows a block of the kernel distance
SMALL = (f" tiny=1 device={DEVICE} probe_data={SMOKE}/rows probe_rows=4 probe_per_band=2 chunk=2 steps=4 batch=2 read_every=2"
         f" sample_steps=4 group={SMOKE_GROUP} save_every=0 log_every=0 pic_data={SMOKE}/judge judge=fake clip=fake sheet_n=4"
         + ("" if DEVICE == "cuda" else " autocast=none"))
smoke_jobs = job_list(f"{SMOKE}/runs", SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP)
smoke_jobs.append(("xchg_self", f"variant=exchange parts=clean=teacher,middle={SMOKE}/runs/middle_tiny_random_s0,noisy=teacher "
                   f"band=middle other={SMOKE}/runs/middle_tiny_random_s0 {DISTANCE}", ["middle_tiny_random_s0"], "exchange"))
S = {}
for t, args, needs, kind in smoke_jobs:
    S[t] = run.main("capture", f"{args} {COMMON}{SMALL}".split() + [f"out={SMOKE}/runs/{t}"])
x = S["xchg_self"]["exchange"]
assert x["final_latent_change"] == 0.0 and all(v["change"] == 0.0 and v["largest_change_where_silent"] == 0.0
                                               for v in x["bands"].values()), "an engine exchanged for itself changed something"
assert all(v["largest_change_where_silent"] == 0.0 for t, *_ , k in smoke_jobs if k == "exchange"
           for v in S[t]["exchange"]["bands"].values() if v["silent_steps"]), "an exchange moved a band where it is silent"
assert all(len(S[t]["curve"]) == 3 for t, _, _, k in smoke_jobs if k == "train"), "a training has no read curve"
print(f"smoke: {len(smoke_jobs)} jobs of every kind ran on the tiny model; the exactness keys hold")

# the real judges on the real model, 16 rows of the judge latents; then the judge's own control: the same pictures
# scored against the captions shifted by one row must score lower than against their own
JUDGE_KIND, CLIP_KIND, STEPS16, REAL_EXTRA = "inception", CLIP, 50, ""
FEATURE_DIM = 2048 if JUDGE_KIND == "inception" else 64
J16_OUT = f"{OUT}/judges16"
shutil.rmtree(J16_OUT, ignore_errors=True)
J16 = run.main("capture", (f"variant=pictures teacher={TEACHER} device={DEVICE} threads=4 parts={TEACHER_PARTS} pic_data={JUDGE} "
                           f"pic_first=0 pic_n=16 group=16 noise_seed=2000 sample_steps={STEPS16} guidance=7.5 judge={JUDGE_KIND} "
                           f"clip={CLIP_KIND} dtype=fp16 part_dtype=fp16 sheet_n=16{REAL_EXTRA}").split() + [f"out={J16_OUT}"])
with np.load(f"{J16_OUT}/pictures.npz") as d:
    feats16, clip16 = d["features"], d["clip"]
assert feats16.shape == (16, FEATURE_DIM) and np.isfinite(feats16).all(), f"the judge's features: {feats16.shape}"
assert np.isfinite(clip16).all(), "a text-to-picture score is not finite"
t16 = load_teacher(TEACHER, device=DEVICE, dtype=torch.float16, vae=True)
t16.vae.float()
x16, ids16 = rows_in_order(JUDGE, 0, 16)
shape16 = (int(t16.unet.config.in_channels), int(t16.unet.config.sample_size), int(t16.unet.config.sample_size))
z16 = pictures.sample_latents(Assembly({b: t16 for b in BANDS}), t16.text_states(ids16), t16.empty_states(1), 2000, group=16,
                              steps=STEPS16, guidance=7.5, latent_shape=shape16, alphas_cumprod=t16.alphas_cumprod)
pics16 = pictures.decode(t16.vae, z16)
clip_judge = pictures.make_clip_judge(CLIP_KIND, DEVICE)
own, shifted = clip_judge.scores(pics16, ids16), clip_judge.scores(pics16, ids16.roll(1, 0))
print(f"real judges, 16 pictures of the model: features {tuple(feats16.shape)}; CLIP against their own captions "
      f"{own.mean():.3f} (the job: {clip16.mean():.3f}), against the captions shifted by one row {shifted.mean():.3f}")
assert abs(float(own.mean()) - float(clip16.mean())) < 0.05, "the cell's pictures are not the job's"
assert shifted.mean() < own.mean(), "the CLIP judge does not score pictures higher against their own captions"
ROW_S = J16["seconds_sampling"] / 16                      # seconds a row of a sampled picture job
print(f"sampling: {ROW_S:.2f} s a row (16 rows, 50 steps, guidance 7.5, half precision)")
del t16, z16, pics16, clip_judge
gc.collect()
torch.cuda.empty_cache()

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    # with the tiny engines' weights, so the upload of large files is checked too
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "three-band assembly, part 2: smoke run", ignore=NOT_SHIPPED), \
        "the smoke folder was not shipped: check the token and the repository before the long run"

# the license copy and a notice, shipped to the root of the results folder before any trained weights
LIC = hf_hub_download(repo_id="CompVis/stable-diffusion-license", repo_type="space", filename="license.txt", local_dir=f"{OUT}/license")
NOTICE = f"{OUT}/notice"
os.makedirs(NOTICE, exist_ok=True)
shutil.copyfile(LIC, f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt")       # byte for byte
lic_bytes = open(f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt", "rb").read()
print("license copy: sha256", hashlib.sha256(lic_bytes).hexdigest(), "|", len(lic_bytes), "bytes")
NOTICE_TEXT = (
    "# Notice: files made from Stable Diffusion 1.5\n\n"
    "The `engine.safetensors` and `engine_avg.safetensors` files in the folders beside this notice are band engines: reduced "
    "copies of the Stable Diffusion 1.5 U-Net (`stable-diffusion-v1-5/stable-diffusion-v1-5`, revision `451f4fe1`). Some start "
    "from that model's weights, and all were trained on its answers or in its latent space, so they are derivatives of that "
    "model.\n\n"
    "Stable Diffusion 1.5 is published under the CreativeML Open RAIL-M license. A copy of that license is in this folder "
    "(`LICENSE_STABLE_DIFFUSION_1_5.txt`); its use restrictions (its Attachment A) travel with these files as they do with the "
    "model.\n\n"
    "These are research artifacts of a measurement, not a finished model: each engine covers one stretch of the noise range and "
    "is read beside the original model (one engine of this folder, the one whose name starts with `whole_`, was trained on "
    "the whole range for comparison).\n")
with open(f"{NOTICE}/NOTICE.md", "w", encoding="utf-8", newline="\n") as fh:
    fh.write(NOTICE_TEXT)
AT_PREFIX = set(list_repo_files(REPO)) if REPO else set()      # part 1 shipped them; shipped again only if missing
if REPO and not {f"{PREFIX}/LICENSE_STABLE_DIFFUSION_1_5.txt", f"{PREFIX}/NOTICE.md"} <= AT_PREFIX:
    assert hub.ship(NOTICE, REPO, PREFIX, "three-band assembly, part 2: license copy and notice"), "the license copy and notice were not shipped"
else:
    print("the license copy and the notice are already in the results folder: not shipped again")

# a timing on this card: 30 steps of tiny at batch 16 on the middle band with the prompt, the trainer's per-step
# lines stamped (the card synchronized); the seconds a step = the median gap of steps 11 to 30
teacher = load_teacher(TEACHER, device=DEVICE, dtype=torch.float32)
source = LatentBatches(TRAIN, seed=0)
SEC = {}
for shape in ("tiny",):
    folder = f"{OUT}/timing/{shape.replace('/', '')}"
    shutil.rmtree(folder, ignore_errors=True)
    stamps = []
    def note(line):
        if line.startswith("step "):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            stamps.append(time.time())
    train_band_engine(teacher, shape, "middle", source, steps=30, batch=16, seed=0, out=folder, text=True, autocast="bf16",
                      save_every=0, log_every=1, log=note)
    SEC[shape] = statistics.median([b - a for a, b in zip(stamps[9:-1], stamps[10:])])
    shutil.rmtree(folder, ignore_errors=True)
    print(f"{shape}: {SEC[shape]:.3f} s a training step at batch 16")
del teacher, source
gc.collect()
torch.cuda.empty_cache()

# each job's cost from those numbers: one forward of 16 rows of the model ~ a quarter of a 2222/nomid/3 step, of an
# engine a quarter of its own step; a read of one part on one band is 64 rows x 4 steps = 16 such forwards; a sampling
# of the 64 rows is 4 groups x 50 steps of 32 rows (guidance); a picture job costs its rows times the seconds a row
# measured above (the decoding and the judges included as a tenth of a second a row); every job also loads the model
TEACHER_OVER_TINY = 0.394 / 0.281     # a 2222/nomid/3 step over a tiny step (the registration's section 5)
F = SEC["tiny"] * TEACHER_OVER_TINY / 4
E = {s: SEC[s] / 4 for s in SEC}
def cost(args, kind):
    a = dict(x.split("=", 1) for x in args.split() if "=" in x)
    if kind == "compare":
        return 15.0
    c = 40 + 3 * 16 * F                                            # the job's start and the model's answers on the three bands
    if kind == "train":
        e = E[a["shape"]]
        c += int(a["steps"]) * SEC[a["shape"]] + 16 * e + (int(a["steps"]) // int(a["read_every"])) * 2 * 16 * e + 3 * 16 * e + 16 * F
    elif kind == "sample":
        c += 3 * 16 * F + 2 * 4 * 50 * 2 * F
    elif kind == "exchange":
        c += 3 * 16 * 4 * F + 2 * 4 * 50 * 2 * F
    else:
        n = int(a["pic_n"])
        c = 40 + n * (0.1 if a.get("source") == "rows" else ROW_S + 0.1)
    return c

JOBS = job_list(RUNS)
COST = {t: cost(a, k) for t, a, _, k in JOBS}
assert len({t for t, *_ in JOBS}) == len(JOBS) and all(n in COST for _, _, ns, _ in JOBS for n in ns)
COUNTS = {k: sum(kk == k for *_, kk in JOBS) for k in ("train", "sample", "exchange", "pictures", "compare")}
assert (COUNTS["train"], COUNTS["sample"] + COUNTS["exchange"], COUNTS["pictures"], COUNTS["compare"]) == (4, 4, 4, 4), COUNTS
INSTRUMENTS = {"seconds_a_step_batch16": SEC, "seconds_a_picture_row": ROW_S,
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "training_rows": TRAIN_ROWS,
               "judge_rows": JUDGE_ROWS, "real_judges_16": {"features": list(feats16.shape), "clip_own": float(own.mean()),
                                                           "clip_shifted_by_one_row": float(shifted.mean())}}
print(f"the job list: {len(JOBS)} jobs ({COUNTS['train']} trainings, {COUNTS['sample'] + COUNTS['exchange']} distance reads, "
      f"{COUNTS['pictures']} picture jobs, {COUNTS['compare']} comparisons), about {sum(COST.values()) / 60:.0f} projected "
      f"minutes on this card with one runner")

One runner works the job list in order (the card is full with one training); a job waits for the jobs whose folders it reads. Each finished job folder is shipped at once to `REPO` (a training with its engine files, a picture job with its features, scores and sheet; only the resume checkpoint is left out), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instrument checks are shipped to `_report_part2`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | capture {a} {COMMON}{EXTRA}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENGINE_FILES = ("engine.safetensors", "engine_avg.safetensors", "engine.json", "train.json")
PICTURE_FILES = ("pictures.npz",)

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "train" and not all(os.path.exists(f"{src}/{f}") for f in ENGINE_FILES):
            continue                                             # its engine files are not on the Hub: it is trained again
        if kind[t] == "pictures" and not all(os.path.exists(f"{src}/{f}") for f in PICTURE_FILES):
            continue                                             # its features are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"three-band assembly, part 2: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    time.sleep(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = capture.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_part2", "three-band assembly, part 2: summary, roster and checks") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")